# LeetCode 2224 — Minimum Number of Operations to Convert Time bằng Python

Link đề: https://leetcode.com/problems/minimum-number-of-operations-to-convert-time/ · Độ khó: **Easy** · Chủ đề: String, Greedy

Skeleton LeetCode:

```python
class Solution:
    def convertTime(self, current: str, correct: str) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Đổi một chuỗi giờ `"HH:MM"` thành **tổng số phút** kể từ 00:00 bằng `split`, `int` và công thức `giờ * 60 + phút`.
2. Hiểu bài toán **đổi tiền (coin change)** với các mệnh giá `60, 15, 5, 1` và vì sao **tham lam (greedy)** đúng ở đây.
3. Dùng `divmod` để vừa lấy thương vừa lấy dư trong một lần.
4. Viết 4 cách: mô phỏng từng bước, tham lam bằng `while`, tham lam bằng `divmod`, và **quy hoạch động / BFS** để kiểm chứng greedy.
5. Trace, test ngẫu nhiên đối chiếu, và tránh các lỗi: quên đổi sang phút, làm tròn sai, nhầm khi giờ khác nhau.

## 2. Đề bài & ví dụ

Cho hai chuỗi `current` và `correct` biểu diễn thời gian **24 giờ** dạng `"HH:MM"`. Mỗi **thao tác** bạn có thể **cộng thêm** `1`, `5`, `15` hoặc `60` phút vào `current` (có thể thực hiện nhiều lần). Trả về **số thao tác ít nhất** để biến `current` thành `correct`.

Đề đảm bảo `current <= correct`, nên không cần xử lý qua nửa đêm.

| Ví dụ | current | correct | Output |
|---|---|---|---|
| 1 | `"02:30"` | `"04:35"` | `3` |
| 2 | `"11:00"` | `"11:01"` | `1` |

- Ví dụ 1: cần thêm `125` phút = `60 + 60 + 5` → 3 thao tác.
- Ví dụ 2: cần thêm `1` phút → 1 thao tác.

Constraints: `current` và `correct` có định dạng `"HH:MM"` hợp lệ; `current <= correct`.

In [1]:
examples = [("02:30", "04:35", 3), ("11:00", "11:01", 1)]
for cur, cor, expected in examples:
    print(cur, "->", cor, "| mong đợi:", expected, "| cách tách:", cur.split(":"))

02:30 -> 04:35 | mong đợi: 3 | cách tách: ['02', '30']
11:00 -> 11:01 | mong đợi: 1 | cách tách: ['11', '00']


## 3. Kiến thức nền

### 3.1. `str.split` và `int`

`"02:30".split(":")` tách chuỗi tại dấu `:` thành `["02", "30"]`. `int("02")` bằng `2` (số 0 đứng đầu được bỏ qua). Ta có thể **gán nhiều biến** cùng lúc: `h, m = ...`.

In [2]:
parts = "02:30".split(":")
print(parts, [int(p) for p in parts])
h, m = map(int, "04:35".split(":"))
print(h, m)
print(int("07"), int("00"))

['02', '30'] [2, 30]
4 35
7 0


### 3.2. Đổi giờ:phút sang tổng số phút

Một giờ có `60` phút nên thời điểm `h:m` cách `00:00` đúng `h * 60 + m` phút. Việc này biến bài toán về **hiệu hai số nguyên** — dễ hơn nhiều so với tính trên giờ và phút riêng rẽ.

In [3]:
def to_minutes(t):
    h, m = map(int, t.split(":"))
    return h * 60 + m

print(to_minutes("00:00"), to_minutes("02:30"), to_minutes("04:35"), to_minutes("23:59"))
print(to_minutes("04:35") - to_minutes("02:30"))

0 150 275 1439
125


### 3.3. `//`, `%` và `divmod`

`a // b` là phép chia lấy **thương nguyên**, `a % b` là **số dư**. `divmod(a, b)` trả về cả hai dưới dạng `(a // b, a % b)`. Ví dụ `125 // 60 = 2`, `125 % 60 = 5`.

In [4]:
print(125 // 60, 125 % 60, divmod(125, 60))
q, r = divmod(37, 15)
print(q, r)

2 5 (2, 5)
2 7


### 3.4. Vòng `for` qua danh sách mệnh giá

Ta lặp qua `(60, 15, 5, 1)` từ lớn đến nhỏ. Ở mỗi mệnh giá, dùng được bao nhiêu thì dùng, phần còn lại chuyển cho mệnh giá nhỏ hơn.

In [5]:
for coin in (60, 15, 5, 1):
    print("mệnh giá", coin, "-> 125 chứa", 125 // coin, "lần")

mệnh giá 60 -> 125 chứa 2 lần
mệnh giá 15 -> 125 chứa 8 lần
mệnh giá 5 -> 125 chứa 25 lần
mệnh giá 1 -> 125 chứa 125 lần


### 3.5. Bài đổi tiền (coin change) và tham lam

Cho các mệnh giá và một số tiền, tìm số đồng ít nhất để trả đúng số tiền đó. **Tham lam** nghĩa là luôn lấy đồng **lớn nhất còn lấy được**. Tham lam **không** luôn đúng (ví dụ mệnh giá `1, 3, 4`, tổng `6`: tham lam cho `4+1+1` = 3 đồng, còn tối ưu là `3+3` = 2 đồng), nhưng đúng khi các mệnh giá có tính chất đặc biệt, như `1, 5, 15, 60`: mỗi mệnh giá lớn là **bội** của mệnh giá nhỏ kế nó (`60` chia hết cho `15`, `15` chia hết cho `5`, `5` chia hết cho `1`).

In [6]:
def greedy_coins(total, coins):
    cnt = 0
    for c in sorted(coins, reverse=True):
        cnt += total // c
        total %= c
    return cnt if total == 0 else None

print(greedy_coins(6, (4, 3, 1)))     # tham lam: 4+1+1 = 3 đồng (không tối ưu)
print(greedy_coins(125, (60, 15, 5, 1)))

3
3


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1: `02:30 → 04:35`.

1. Đổi sang phút: `02:30 = 150`, `04:35 = 275`. Cần cộng thêm `275 - 150 = 125` phút.
2. Dùng mệnh giá **to nhất trước**: `125 = 2 × 60 + 5` → hai lần `+60`, còn `5`.
3. `5 = 1 × 5` → một lần `+5`, còn `0`.
4. Tổng `2 + 1 = 3` thao tác.

Trực giác: thao tác `+60` "đi xa" nhất nên dùng nhiều nhất có thể; phần dư nhỏ hơn `60` lại dùng `+15` nhiều nhất có thể, và cứ thế.

In [7]:
diff = 125
steps = []
for coin in (60, 15, 5, 1):
    k, diff = divmod(diff, coin)
    steps.append((coin, k))
print(steps, "tổng thao tác =", sum(k for _, k in steps))

[(60, 2), (15, 0), (5, 1), (1, 0)] tổng thao tác = 3


## 5. Cách 1 — Brute Force / baseline: mô phỏng từng thao tác bằng `while`

Cho `cur` chạy từ `current` đến `correct`. Mỗi bước, thử `+60`; nếu vượt quá `correct` thì thử `+15`, rồi `+5`, rồi `+1`. Đây là tham lam dưới dạng mô phỏng trực tiếp, mỗi vòng chỉ một thao tác.

In [8]:
def convert_sim(current, correct):
    def to_min(t):
        h, m = map(int, t.split(":"))
        return h * 60 + m
    cur, target = to_min(current), to_min(correct)
    ops = 0
    while cur < target:
        for step in (60, 15, 5, 1):
            if cur + step <= target:
                cur += step
                ops += 1
                break
    return ops

print(convert_sim("02:30", "04:35"), convert_sim("11:00", "11:01"))

3 1


In [9]:
from typing import List

class SolutionSimulate:
    def convertTime(self, current: str, correct: str) -> int:
        h1, m1 = map(int, current.split(":"))
        h2, m2 = map(int, correct.split(":"))
        cur = h1 * 60 + m1
        target = h2 * 60 + m2
        ops = 0
        while cur < target:
            for step in (60, 15, 5, 1):
                if cur + step <= target:
                    cur += step
                    ops += 1
                    break
        return ops

print(SolutionSimulate().convertTime("02:30", "04:35"))
print(SolutionSimulate().convertTime("11:00", "11:01"))

3
1


**Độ phức tạp:** Gọi `D = target - cur` (số phút cần thêm, tối đa `1439`). Mỗi vòng `while` thêm ít nhất `1` phút nên chạy tối đa `D` lần, nhưng thực tế chỉ tốn số thao tác tối ưu (≤ `23 + 3 + 2 + 4 = 32`). **Time `O(D)` xấu nhất**, **Space `O(1)`**.

**Nút thắt:** ta mô phỏng từng bước một dù có thể tính bằng chia trực tiếp. Biết `D` rồi, mỗi mệnh giá chỉ cần **một phép chia**.

## 6. Ý tưởng thuật toán / bước đột phá

Insight: sau khi đổi sang phút, bài toán là **đổi `D` phút bằng các đồng `60, 15, 5, 1`** sao cho dùng ít đồng nhất. Quy tắc tham lam:

- nếu còn `D ≥ 60` thì dùng `D // 60` đồng `60`, đặt `D = D % 60`;
- làm tương tự lần lượt với `15`, `5`, `1`;
- tham lam **đúng** vì mỗi mệnh giá là **bội** của mệnh giá nhỏ hơn kế nó (`60 = 4·15`, `15 = 3·5`, `5 = 5·1`). Hai đồng nhỏ nào cũng có thể gộp lại thành một đồng lớn đúng bằng tổng, nên không bao giờ thiệt khi dùng đồng lớn trước.

Chỉ cần bốn phép `divmod`, độ phức tạp `O(1)`.

## 7. Các cách tiếp theo

### Cách 2 — Tham lam bằng `divmod` trên tổng số phút chênh lệch

Tính `D = to_min(correct) - to_min(current)`, rồi với mỗi mệnh giá cộng thương vào đáp án và giữ lại số dư.

In [10]:
from typing import List

class SolutionGreedy:
    def convertTime(self, current: str, correct: str) -> int:
        h1, m1 = map(int, current.split(":"))
        h2, m2 = map(int, correct.split(":"))
        diff = (h2 * 60 + m2) - (h1 * 60 + m1)
        ops = 0
        for coin in (60, 15, 5, 1):
            k, diff = divmod(diff, coin)
            ops += k
        return ops

print(SolutionGreedy().convertTime("02:30", "04:35"))
print(SolutionGreedy().convertTime("11:00", "11:01"))
print(SolutionGreedy().convertTime("00:00", "23:59"))

3
1
32


**Độ phức tạp:** Time `O(1)` (luôn đúng 4 phép chia), Space `O(1)`.

### Cách 3 — Chia không dùng `divmod`, viết tường minh

Giống cách 2 nhưng tách từng bước, rất tiện để in debug và chỉ ra "mỗi mệnh giá lấy bao nhiêu".

In [11]:
from typing import List

class SolutionExplicit:
    def convertTime(self, current: str, correct: str) -> int:
        a = int(current[:2]) * 60 + int(current[3:])
        b = int(correct[:2]) * 60 + int(correct[3:])
        d = b - a
        ops = d // 60
        d %= 60
        ops += d // 15
        d %= 15
        ops += d // 5
        d %= 5
        ops += d
        return ops

print(SolutionExplicit().convertTime("02:30", "04:35"))
print(SolutionExplicit().convertTime("11:00", "11:01"))
cur = "09:41"
print(cur, int(cur[:2]), int(cur[3:]))

3
1
09:41 9 41


**Độ phức tạp:** Time `O(1)`, Space `O(1)`. Lưu ý lát cắt cố định `[:2]` và `[3:]` an toàn vì định dạng luôn là `HH:MM` đúng 5 ký tự.

### Cách 4 — Quy hoạch động (đổi tiền kinh điển) để **kiểm chứng** greedy

Gọi `dp[x]` là số thao tác ít nhất để cộng đúng `x` phút. `dp[0] = 0`, và `dp[x] = 1 + min(dp[x - c])` với mọi `c ∈ {1, 5, 15, 60}` và `x - c ≥ 0`. Cách này **không** dựa vào giả thiết tham lam nên là "thước đo chuẩn" để đối chiếu.

In [12]:
from typing import List

class SolutionDP:
    def convertTime(self, current: str, correct: str) -> int:
        h1, m1 = map(int, current.split(":"))
        h2, m2 = map(int, correct.split(":"))
        diff = (h2 * 60 + m2) - (h1 * 60 + m1)
        INF = float("inf")
        dp = [0] + [INF] * diff
        for x in range(1, diff + 1):
            for c in (1, 5, 15, 60):
                if x >= c and dp[x - c] + 1 < dp[x]:
                    dp[x] = dp[x - c] + 1
        return dp[diff]

print(SolutionDP().convertTime("02:30", "04:35"))
print(SolutionDP().convertTime("11:00", "11:01"))
print(SolutionDP().convertTime("00:00", "23:59"))

3
1
32


**Độ phức tạp:** Time `O(D · 4) = O(D)` với `D ≤ 1439`, Space `O(D)`. Chậm hơn tham lam nhưng **luôn đúng với mọi bộ mệnh giá**; ta dùng nó ở mục test để chứng minh greedy đúng cho bài này (trên toàn bộ các giá trị `D` từ `0` đến `1439`).

## 8. TRACE từng bước

Trace Cách 2 cho `current = "02:30"`, `correct = "04:35"`:

| bước | mệnh giá `coin` | `diff` trước | `k = diff // coin` | `diff` sau | `ops` |
|---|---|---|---|---|---|
| bắt đầu | – | 125 | – | – | 0 |
| 1 | 60 | 125 | 2 | 5 | 2 |
| 2 | 15 | 5 | 0 | 5 | 2 |
| 3 | 5 | 5 | 1 | 0 | 3 |
| 4 | 1 | 0 | 0 | 0 | 3 |

Kết quả `3`.

In [13]:
def trace_greedy(current, correct):
    h1, m1 = map(int, current.split(":"))
    h2, m2 = map(int, correct.split(":"))
    a, b = h1 * 60 + m1, h2 * 60 + m2
    diff = b - a
    print(f"{current} = {a} phút | {correct} = {b} phút | chênh lệch = {diff}")
    ops = 0
    for coin in (60, 15, 5, 1):
        k, rest = divmod(diff, coin)
        ops += k
        print(f"  coin={coin:2d}: diff={diff:4d} -> k={k}, còn lại={rest}, ops={ops}")
        diff = rest
    return ops

print("kết quả:", trace_greedy("02:30", "04:35"))
print("kết quả:", trace_greedy("11:00", "11:01"))

02:30 = 150 phút | 04:35 = 275 phút | chênh lệch = 125
  coin=60: diff= 125 -> k=2, còn lại=5, ops=2
  coin=15: diff=   5 -> k=0, còn lại=5, ops=2
  coin= 5: diff=   5 -> k=1, còn lại=0, ops=3
  coin= 1: diff=   0 -> k=0, còn lại=0, ops=3
kết quả: 3
11:00 = 660 phút | 11:01 = 661 phút | chênh lệch = 1
  coin=60: diff=   1 -> k=0, còn lại=1, ops=0
  coin=15: diff=   1 -> k=0, còn lại=1, ops=0
  coin= 5: diff=   1 -> k=0, còn lại=1, ops=0
  coin= 1: diff=   1 -> k=1, còn lại=0, ops=1
kết quả: 1


In [14]:
def trace_sim(current, correct):
    h1, m1 = map(int, current.split(":"))
    h2, m2 = map(int, correct.split(":"))
    cur, target = h1 * 60 + m1, h2 * 60 + m2
    ops = 0
    while cur < target:
        for step in (60, 15, 5, 1):
            if cur + step <= target:
                cur += step
                ops += 1
                print(f"  +{step:2d} -> cur={cur} ({cur // 60:02d}:{cur % 60:02d}), ops={ops}")
                break
    return ops

trace_sim("02:30", "04:35")

  +60 -> cur=210 (03:30), ops=1
  +60 -> cur=270 (04:30), ops=2
  + 5 -> cur=275 (04:35), ops=3


3

## 9. Code nộp LeetCode

In [15]:
from typing import List

class Solution:
    def convertTime(self, current: str, correct: str) -> int:
        h1, m1 = map(int, current.split(":"))
        h2, m2 = map(int, correct.split(":"))
        diff = (h2 * 60 + m2) - (h1 * 60 + m1)
        ops = 0
        for coin in (60, 15, 5, 1):
            k, diff = divmod(diff, coin)
            ops += k
        return ops

print(Solution().convertTime("02:30", "04:35"))
print(Solution().convertTime("11:00", "11:01"))

3
1


## 10. Vì sao thuật toán đúng

**Bước đổi sang phút:** `h * 60 + m` là một ánh xạ **một-một** từ `HH:MM` (với `0 ≤ m < 60`) sang số nguyên `0..1439`, và cộng `x` phút vào giờ tương ứng cộng `x` vào số nguyên. Nên số thao tác cần thiết chỉ phụ thuộc vào `D = target - cur`.

**Tham lam đúng (lập luận trao đổi):** xét một nghiệm tối ưu bất kỳ.

- Nếu dùng `≥ 5` đồng `1`, đổi `5` đồng `1` thành `1` đồng `5` → ít đồng hơn, mâu thuẫn. Vậy tối ưu có `≤ 4` đồng `1`.
- Nếu dùng `≥ 3` đồng `5`, đổi thành `1` đồng `15` → `≤ 2` đồng `5`.
- Nếu dùng `≥ 4` đồng `15`, đổi thành `1` đồng `60` → `≤ 3` đồng `15`.
- Tổng của phần "nhỏ" (không tính đồng `60`) tối đa là `4·1 + 2·5 + 3·15 = 59 < 60`, nên số đồng `60` **bị xác định**: đúng `D // 60`.
- Phần dư `R = D % 60`: vì `5` và `15` đều chia hết cho `5`, số đồng `1` buộc bằng `R % 5` (vì `≤ 4`); phần còn lại là bội của `5`, số đồng `5` bị ép bằng `(R // 5) % 3` (vì `≤ 2`) và số đồng `15` là phần còn lại.

Mọi con số đều bị xác định duy nhất, và đó chính là kết quả của `divmod` liên tiếp — tức **tham lam là nghiệm tối ưu duy nhất**.

**Kiểm chứng bằng DP:** ở mục test ta đối chiếu tham lam và DP với mọi `D ∈ [0, 1439]` — hai kết quả luôn bằng nhau.

## 11. Phân tích Time / Space Complexity

Đặt `D` = số phút chênh lệch (`0 ≤ D ≤ 1439`).

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng `while` | `O(số thao tác)` ≤ `O(D)` | `O(1)` |
| 2. Tham lam `divmod` | `O(1)` | `O(1)` |
| 3. Chia tường minh | `O(1)` | `O(1)` |
| 4. DP đổi tiền | `O(D)` | `O(D)` |

Vì `D` bị chặn bởi `1439` nên mọi cách đều rất nhanh. Về mặt khái niệm, cách 2/3 là `O(1)` thật sự (số phép chia cố định).

## 12. Bộ test

Ví dụ đề, edge case (cùng giờ, qua nhiều giờ, `00:00 → 23:59`), phản ví dụ greedy, và đối chiếu toàn bộ cặp `(current, correct)` ngẫu nhiên.

In [16]:
import random

solvers = [SolutionSimulate(), SolutionGreedy(), SolutionExplicit(), SolutionDP(), Solution()]

fixed = [
    ("02:30", "04:35", 3),
    ("11:00", "11:01", 1),
    ("00:00", "00:00", 0),                    # không cần thao tác
    ("00:00", "23:59", 23 + 3 + 2 + 4),       # 23*60 + 3*15 + 2*5 + 4
    ("09:15", "10:15", 1),                    # đúng một giờ
    ("12:59", "13:00", 1),                    # qua mốc giờ nhưng chỉ cần 1 phút
    ("01:00", "01:14", 6),                     # 14 phút = 2*5 + 4*1 (chưa dùng được đồng 15)
]
for cur, cor, expected in fixed:
    for sol in solvers:
        assert sol.convertTime(cur, cor) == expected, (type(sol).__name__, cur, cor, sol.convertTime(cur, cor), expected)
print("test cố định: OK")

# Greedy đúng với MỌI chênh lệch D từ 0..1439 (so với DP)
def fmt(x):
    return f"{x // 60:02d}:{x % 60:02d}"
for D in range(0, 1440):
    assert SolutionGreedy().convertTime("00:00", fmt(D)) == SolutionDP().convertTime("00:00", fmt(D)), D
print("greedy == DP với mọi D trong [0, 1439]: OK")

random.seed(2224)
for _ in range(500):
    a = random.randint(0, 1439)
    b = random.randint(a, 1439)
    cur, cor = fmt(a), fmt(b)
    res = {type(s).__name__: s.convertTime(cur, cor) for s in solvers}
    assert len(set(res.values())) == 1, (cur, cor, res)
print("500 test ngẫu nhiên: OK")

test cố định: OK


greedy == DP với mọi D trong [0, 1439]: OK
500 test ngẫu nhiên: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: So sánh giờ và phút riêng rẽ

Nhiều bạn tính `giờ_chênh * 4 + ...` mà quên `phút` có thể âm (ví dụ `09:50 → 10:10` thì chênh phút là `10 - 50 = -40`). Đổi sang tổng số phút trước thì khỏi lỗi.

In [17]:
def wrong1(current, correct):
    h1, m1 = map(int, current.split(":"))
    h2, m2 = map(int, correct.split(":"))
    dh, dm = h2 - h1, m2 - m1          # dm có thể âm!
    ops = dh                            # dùng +60 cho phần giờ
    for coin in (15, 5, 1):
        k, dm = divmod(dm, coin)
        ops += k
    return ops

print("sai :", wrong1("09:50", "10:10"), "| dm = -40 làm kết quả vô lý")
print("đúng:", Solution().convertTime("09:50", "10:10"), "(20 phút = 15 + 5 = 2 thao tác)")

sai : -1 | dm = -40 làm kết quả vô lý
đúng: 2 (20 phút = 15 + 5 = 2 thao tác)


### Lỗi 2: Tham lam theo thứ tự tăng dần (`1, 5, 15, 60`)

Lấy đồng nhỏ trước sẽ dùng `1` cho tất cả → số thao tác rất lớn.

In [18]:
def wrong2(diff):
    ops = 0
    for coin in (1, 5, 15, 60):
        k, diff = divmod(diff, coin)
        ops += k
    return ops

print("sai :", wrong2(125), "| đúng:", Solution().convertTime("00:00", "02:05"))

sai : 125 | đúng: 3


### Lỗi 3: Dùng `diff / coin` (chia thực) thay vì `//`

`125 / 60 = 2.0833...` là số thực; cộng vào `ops` làm đáp án sai kiểu và sai giá trị.

In [19]:
diff = 125
print("sai :", diff / 60, type(diff / 60))
print("đúng:", diff // 60, type(diff // 60))

sai : 2.0833333333333335 <class 'float'>
đúng: 2 <class 'int'>


### Lỗi 4: Quên phần dư ở mệnh giá cuối

Nếu thiếu mệnh giá `1` thì số dư không được "tiêu hết" và đáp án thiếu.

In [20]:
def wrong4(diff):
    ops = 0
    for coin in (60, 15, 5):               # thiếu 1
        k, diff = divmod(diff, coin)
        ops += k
    return ops, diff                        # diff còn dư > 0

print("sai :", wrong4(7), "(còn dư 2 phút chưa xử lý)")
print("đúng:", Solution().convertTime("00:00", "00:07"))

sai : (1, 2) (còn dư 2 phút chưa xử lý)
đúng: 3


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chuỗi giờ `HH:MM`" → đổi ngay sang **số phút** để biến thành bài số học.
- "Số thao tác ít nhất với bước `1, 5, 15, 60`" → **đổi tiền**; kiểm tra tính chia hết giữa các mệnh giá để biết greedy có đúng không. Bộ như `1, 3, 4` thì **không** dùng greedy.
- `k, d = divmod(d, coin)` trong vòng lặp qua mệnh giá giảm dần là khuôn mẫu rất hay dùng.
- Nếu nghi ngờ greedy, **viết thêm DP** và so sánh trên toàn bộ miền nhỏ (như ta làm với `D ≤ 1439`).
- Khi đọc đề, chú ý có cho qua nửa đêm hay không (ở đây không vì `current <= correct`).

## 15. Biến thể của bài

1. Cho phép **qua nửa đêm** (giờ quay vòng 24h): dùng `(target - cur) % 1440`.
2. Cho phép **cộng hoặc trừ** thao tác: dùng BFS trên các thời điểm.
3. Thay bộ thao tác bằng bộ khác (ví dụ `1, 3, 4`): greedy sai, phải dùng DP (LeetCode 322 Coin Change).
4. Đếm **số cách** (không phải ít nhất) để đạt `D` (LeetCode 518 Coin Change II).
5. Cho nhiều đồng hồ, tìm tổng thao tác tối thiểu để đồng bộ tất cả với một mốc chuẩn.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng `while` | cộng từng bước tham lam | `O(D)` | `O(1)` | hiểu quá trình, debug từng bước |
| 2. Tham lam `divmod` | 4 phép chia | `O(1)` | `O(1)` | **khuyến nghị khi nộp** |
| 3. Chia tường minh | khai triển từng bước | `O(1)` | `O(1)` | ngôn ngữ không có `divmod`, muốn rõ ràng |
| 4. DP đổi tiền | `dp[x] = 1 + min(dp[x-c])` | `O(D)` | `O(D)` | bộ mệnh giá bất kỳ, hoặc để kiểm chứng greedy |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** `print(divmod(100, 60), 100 // 15, 100 % 15)` in ra gì?

**Bài 2 (tính tay).** `current = "08:20"`, `correct = "11:47"`. Tính số thao tác tối thiểu.

**Bài 3 (điền TODO).** Hoàn thành hàm đổi `"HH:MM"` thành số phút.

In [21]:
def to_minutes(t):
    # TODO: tách bằng split(":"), đổi int, trả về h * 60 + m
    pass

# chưa gọi hàm ở đây để notebook chạy không lỗi

**Bài 4 (tự viết lại).** Viết `convert_wrap(current, correct)` cho phép **qua nửa đêm**: nếu `correct < current` thì hiểu là ngày hôm sau. Trả về số thao tác tối thiểu.

In [22]:
def convert_wrap(current, correct):
    # TODO: diff = (target - cur) % 1440 rồi tham lam với 60, 15, 5, 1
    pass

**Bài 5 (mini challenge).** Cho mệnh giá bất kỳ `coins` (có chứa `1`). Viết `min_ops(diff, coins)` bằng **DP** trả về số đồng ít nhất, và chứng minh bằng ví dụ rằng greedy sai với `coins = (1, 3, 4)`, `diff = 6`.

In [23]:
def min_ops(diff, coins):
    # TODO: dp[0] = 0; dp[x] = 1 + min(dp[x - c])
    pass

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `divmod(100,60) = (1, 40)`; `100 // 15 = 6`; `100 % 15 = 10`. Bài 2: đổi sang phút, trừ nhau được `207`.
- **Mức 2 (vừa):** Bài 2: `207 = 3 × 60 + 27`; `27 = 1 × 15 + 12`; `12 = 2 × 5 + 2`; `2 = 2 × 1`. Bài 3: `h, m = map(int, t.split(":"))`. Bài 4: dùng `% 1440`.
- **Mức 3 (sâu):** Bài 5: `dp = [0] + [inf] * diff`, vòng ngoài `x`, vòng trong `c`. Với `(1,3,4)` và `6`: DP cho `2` (3+3) còn greedy cho `3` (4+1+1).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [24]:
# Bài 1 và 2
assert (divmod(100, 60), 100 // 15, 100 % 15) == ((1, 40), 6, 10)
assert Solution().convertTime("08:20", "11:47") == 3 + 1 + 2 + 2

# Bài 3
def to_minutes(t):
    h, m = map(int, t.split(":"))
    return h * 60 + m

# Bài 4
def convert_wrap(current, correct):
    diff = (to_minutes(correct) - to_minutes(current)) % 1440
    ops = 0
    for coin in (60, 15, 5, 1):
        k, diff = divmod(diff, coin)
        ops += k
    return ops

# Bài 5
def min_ops(diff, coins):
    INF = float("inf")
    dp = [0] + [INF] * diff
    for x in range(1, diff + 1):
        for c in coins:
            if x >= c and dp[x - c] + 1 < dp[x]:
                dp[x] = dp[x - c] + 1
    return dp[diff]

assert to_minutes("00:00") == 0 and to_minutes("23:59") == 1439
assert convert_wrap("23:50", "00:10") == 2          # 20 phút = 15 + 5
assert convert_wrap("02:30", "04:35") == 3
assert min_ops(125, (1, 5, 15, 60)) == 3
assert min_ops(6, (1, 3, 4)) == 2                    # greedy cho 3 (4+1+1)
print("tất cả bài tập OK")

tất cả bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đổi sang phút rồi tham lam bằng `divmod`)

```python
class Solution:
    def convertTime(self, current: str, correct: str) -> int:
        h1, m1 = map(int, current.split(":"))
        h2, m2 = map(int, correct.split(":"))
        diff = (h2 * 60 + m2) - (h1 * 60 + m1)
        ops = 0
        for coin in (60, 15, 5, 1):
            k, diff = divmod(diff, coin)
            ops += k
        return ops
```

**Checklist**

- [ ] Đã đổi cả hai chuỗi sang **tổng số phút** trước khi trừ?
- [ ] Mệnh giá được duyệt **từ lớn đến nhỏ** (`60, 15, 5, 1`)?
- [ ] Dùng `//` / `divmod`, không dùng `/`?
- [ ] Có mệnh giá `1` để tiêu hết số dư?
- [ ] Đã thử `current == correct` (kết quả `0`) và `00:00 → 23:59`?
- [ ] Đã nhớ lý do greedy đúng (mỗi mệnh giá là bội của mệnh giá nhỏ hơn)?
- [ ] Signature đúng `convertTime(self, current: str, correct: str) -> int`?